# QC V03 Microgrid Execution

6 A31 configs, fixed A32, serial vs parallel (jobs=4).
A3=open_macro_v03, A4=harness_ready_provisional_A3, A5=blocked

In [ ]:
from pathlib import Path
import importlib.metadata
import json, os, platform, sys, time, tracemalloc, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from qc_a3_core import (
    A3ParityConfig, materialize_harness_source_from_manifest,
    read_json, write_json, read_npz_records,
    load_a32_config, load_l2_macro_for_config, load_revision_uncertainty_for_config,
)

# --- Bundle setup ---
OBJECT_STORE_MANIFEST_KEY = "investintell/a3/qc-a3-parity/1138754/d9f029f7a4d450ea396504ad/object_store_manifest.json"
RESEARCH_NODE = os.environ.get("QC_NODE_TYPE") or os.environ.get("QC_NODE_NAME") or "qc_research"
PLATFORM_TEXT = platform.platform()

from QuantConnect import TimeZones
from QuantConnect.Research import QuantBook
qb = QuantBook()
qb.set_start_date(2014, 2, 19)
qb.set_end_date(2026, 6, 24)
qb.set_time_zone(TimeZones.NEW_YORK)
object_store = getattr(qb, "object_store", getattr(qb, "ObjectStore", None))

def object_store_file_path(key):
    if hasattr(object_store, "get_file_path"):
        return Path(object_store.get_file_path(key))
    return Path(object_store.GetFilePath(key))

t_env = time.perf_counter()
manifest_path = object_store_file_path(OBJECT_STORE_MANIFEST_KEY)
manifest = read_json(manifest_path)
manifest["_local_bundle_dir"] = str(manifest_path.parent)
assert manifest["a3_status"] == "open_macro_v03"
materialize_harness_source_from_manifest(manifest, object_store_file_path, project_root=PROJECT_ROOT)
env_init_seconds = time.perf_counter() - t_env

try:
    subprocess.run(["git", "init"], capture_output=True, text=True)
    subprocess.run(["git", "add", "-A"], capture_output=True, text=True)
    subprocess.run(["git", "commit", "-m", "init"], capture_output=True, text=True, timeout=30)
except Exception:
    pass

bundle_dir = manifest_path.parent
def artifact_path(name):
    item = manifest["object_files"][name]
    p = object_store_file_path(item["object_store_key"])
    return p if p is not None else bundle_dir / item["relative_path"]

fm_path = artifact_path("feature_manifest")
ru_path = artifact_path("revision_uncertainty_manifest")
cc_path = artifact_path("config_catalog_normalized")
l2_npz = artifact_path("macro_l2_union_numeric")
ru_npz = artifact_path("revision_uncertainty_numeric")

from src import calibration_harness as harness

# --- Cold load from Object Store ---
t_cold = time.perf_counter()
cfg0 = A3ParityConfig(feature_manifest=fm_path, revision_uncertainty_manifest=ru_path,
    config_catalog=cc_path, a32_grid_dir=fm_path.parent, output_dir=Path("results"),
    macro_l2_npz=l2_npz, revision_uncertainty_npz=ru_npz,
    a31_name="V03-G0-CONTROL", a32_name="A32-G0.35-I0.35-X0.10-C0.60-D1.25",
    worker_commit=manifest["worker_commit"])
_fm, _l2p, l2_hash, l2_records = load_l2_macro_for_config(cfg0)
_rum, ru_hash, ru_records = load_revision_uncertainty_for_config(cfg0)
cold_load_seconds = time.perf_counter() - t_cold

# --- Bundle decode ---
t_decode = time.perf_counter()
ru_by_key = harness.revision_uncertainty_keyed(ru_records)
catalog_payload = harness.read_catalog_payload(cc_path)
norm_catalog, catalog_hash = harness.normalize_a31_catalog(
    catalog_payload, l2_macro_logical_hash=l2_hash, source_path=cc_path)
decode_seconds = time.perf_counter() - t_decode

# --- Warm load (re-read from local disk) ---
t_warm = time.perf_counter()
l2_warm = read_npz_records(l2_npz)
ru_warm = read_npz_records(ru_npz)
warm_load_seconds = time.perf_counter() - t_warm

A32_NAME = "A32-G0.35-I0.35-X0.10-C0.60-D1.25"
TARGET_A31 = ["V03-G0-CONTROL", "V03-G1-FAMILY-WEIGHTED-MEDIAN", "V03-G1-REVSOFT-P50",
    "V03-G1-REVSOFT-P75", "V03-G1-FAMILY-CONSENSUS-60", "V03-G1-FAMILY-CONSENSUS-67"]
FOLDS = ["full", "post_initialization", "2014_2017", "2018_2021", "2022_2026"]

def run_config(a31_name):
    t0 = time.perf_counter()
    tracemalloc.start()
    a31_item = [i for i in norm_catalog["configs"] if i["config"]["name"] == a31_name][0]
    a31 = harness.A31Config(**a31_item["config"])
    a31h = str(a31_item["a31_config_hash"])
    t_l3 = time.perf_counter()
    l3, contrib, l3m = harness.build_l3_score_panel(
        l2_records, a31, l2_macro_logical_hash=l2_hash,
        expected_l2_macro_logical_hash=l2_hash,
        revision_uncertainty_by_key=ru_by_key,
        revision_uncertainty_logical_hash=ru_hash)
    dt_l3 = time.perf_counter() - t_l3
    a32 = load_a32_config(fm_path.parent, A32_NAME)
    a32h = harness.a32_config_hash(a32)
    eh = harness.evaluation_hash(a31h, a32h)
    t_l4 = time.perf_counter()
    rt, _ = harness.run_l4_state_machine(l3, a32, selection_mode="latest")
    cf, _ = harness.run_l4_state_machine(l3, a32, selection_mode="first_release")
    dt_l4 = time.perf_counter() - t_l4
    t_m = time.perf_counter()
    mf = harness.build_macro_metrics(rt, first_release_replay=cf)
    cls = harness.classify_a32_grid_result(mf)
    mr = harness.evaluation_metric_rows(rt, cf, a31, a32, a31h, a32h, eh, cls)
    dt_m = time.perf_counter() - t_m
    rt_hash = harness.logical_records_hash(rt)
    cf_hash = harness.logical_records_hash(cf)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        "a31_config_name": a31_name, "a31_config_hash": a31h,
        "a32_config_name": A32_NAME, "a32_config_hash": a32h,
        "evaluation_hash": eh, "classification": cls,
        "runtime_row_count": len(rt), "counterfactual_row_count": len(cf),
        "metric_row_count": len(mr),
        "runtime_replay_logical_hash": rt_hash,
        "counterfactual_replay_logical_hash": cf_hash,
        "metrics_by_fold": {r["fold"]: r for r in mr},
        "metrics_full": mf,
        "timings": {"compute_l3": dt_l3, "run_l4": dt_l4, "compute_metrics": dt_m,
                   "total": time.perf_counter() - t0},
        "peak_memory_bytes": peak,
    }

print(f"Setup complete. env_init={env_init_seconds:.3f}s, cold_load={cold_load_seconds:.3f}s, "
      f"decode={decode_seconds:.3f}s, warm_load={warm_load_seconds:.3f}s")
print(f"L2 records: {len(l2_records)}, uncertainty records: {len(ru_records)}")
print(f"Configs: {len(TARGET_A31)}, A32: {A32_NAME}")


In [ ]:
# --- Serial execution (jobs=1) ---
serial_results = []
t_serial = time.perf_counter()
tracemalloc.start()
for i, name in enumerate(TARGET_A31):
    r = run_config(name)
    serial_results.append(r)
    print(f"  S[{i+1}/6] {name}: cls={r['classification']}, rt_hash={r['runtime_replay_logical_hash'][:12]}...,")
    print(f"         cf_hash={r['counterfactual_replay_logical_hash'][:12]}..., time={r['timings']['total']:.1f}s")
_, serial_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
serial_total = time.perf_counter() - t_serial
print(f"\nSerial total: {serial_total:.1f}s, peak_mem: {serial_peak/1e6:.1f}MB")

# --- Parallel execution (jobs=4) ---
parallel_results = []
t_parallel = time.perf_counter()
tracemalloc.start()
print(f"\nStarting parallel (jobs=4)...")
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(run_config, name): name for name in TARGET_A31}
    for fut in as_completed(futures):
        name = futures[fut]
        r = fut.result()
        parallel_results.append(r)
        print(f"  P DONE: {name}: cls={r['classification']}, rt_hash={r['runtime_replay_logical_hash'][:12]}...,")
        print(f"         cf_hash={r['counterfactual_replay_logical_hash'][:12]}..., time={r['timings']['total']:.1f}s")
_, parallel_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
parallel_total = time.perf_counter() - t_parallel
parallel_results.sort(key=lambda r: TARGET_A31.index(r["a31_config_name"]))
serial_results.sort(key=lambda r: TARGET_A31.index(r["a31_config_name"]))
print(f"\nParallel total: {parallel_total:.1f}s, peak_mem: {parallel_peak/1e6:.1f}MB")
print(f"Speedup: {serial_total / parallel_total:.2f}x")


In [ ]:
# --- Comparison and output generation ---
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# 1. Compare serial vs parallel
parity_check = {"status": "passed", "mismatches": [], "config_count": len(TARGET_A31)}
for s, p in zip(serial_results, parallel_results):
    name = s["a31_config_name"]
    checks = {
        "runtime_hash_match": s["runtime_replay_logical_hash"] == p["runtime_replay_logical_hash"],
        "counterfactual_hash_match": s["counterfactual_replay_logical_hash"] == p["counterfactual_replay_logical_hash"],
        "classification_match": s["classification"] == p["classification"],
        "runtime_row_count_match": s["runtime_row_count"] == p["runtime_row_count"],
        "metric_row_count_match": s["metric_row_count"] == p["metric_row_count"],
    }
    if not all(checks.values()):
        parity_check["status"] = "failed"
        parity_check["mismatches"].append({"config": name, "checks": checks})
    for fold in FOLDS:
        s_fold = s["metrics_by_fold"].get(fold, {})
        p_fold = p["metrics_by_fold"].get(fold, {})
        for key in set(s_fold) | set(p_fold):
            sv, pv = s_fold.get(key), p_fold.get(key)
            if sv != pv and key not in ("fold",):
                parity_check["status"] = "failed"
                parity_check["mismatches"].append({"config": name, "fold": fold, "field": key})

print(f"Serial vs Parallel parity: {parity_check['status']}")
print(f"Mismatches: {len(parity_check['mismatches'])}")

# 2. Build summary and metrics outputs
def extract_key_metrics(result, fold="full"):
    m = result["metrics_by_fold"].get(fold, {})
    return {
        "a31_config_name": result["a31_config_name"],
        "classification": result["classification"],
        "abstain_rate": m.get("abstain_rate"),
        "valid_rate": m.get("valid_rate"),
        "neutral_rate": m.get("neutral_rate"),
        "candidate_flips_per_year": m.get("candidate_flips_per_year"),
        "published_flips_per_year": m.get("published_flips_per_year"),
        "candidate_quadrant_change_rate_calendar": m.get("candidate_quadrant_change_rate_calendar"),
        "status_revision_change_rate": m.get("status_revision_change_rate"),
        "runtime_replay_logical_hash": result["runtime_replay_logical_hash"],
        "counterfactual_replay_logical_hash": result["counterfactual_replay_logical_hash"],
    }

serial_summary = [extract_key_metrics(r) for r in serial_results]
parallel_summary = [extract_key_metrics(r) for r in parallel_results]

# 3. Pareto analysis
def is_pareto_dominated(candidate, others):
    c_abs = candidate.get("abstain_rate", 1) or 1
    c_flips = abs(candidate.get("published_flips_per_year", 999) or 999)
    for o in others:
        o_abs = o.get("abstain_rate", 1) or 1
        o_flips = abs(o.get("published_flips_per_year", 999) or 999)
        if o_abs <= c_abs and o_flips <= c_flips and (o_abs < c_abs or o_flips < c_flips):
            return True
    return False

pareto_front = []
for i, r in enumerate(serial_summary):
    others = [o for j, o in enumerate(serial_summary) if j != i]
    if not is_pareto_dominated(r, others):
        pareto_front.append(r["a31_config_name"])

# 4. Final ordering
final_order = sorted(serial_summary, key=lambda r: (
    r.get("abstain_rate", 1) or 1,
    abs(r.get("published_flips_per_year", 999) or 999)))
final_order_names = [r["a31_config_name"] for r in final_order]

print(f"Pareto front: {pareto_front}")
print(f"Final ordering: {final_order_names}")

# 5. Performance summary
perf = {
    "environment_init": env_init_seconds,
    "object_store_cold_load": cold_load_seconds,
    "object_store_warm_load": warm_load_seconds,
    "bundle_decode": decode_seconds,
    "serial_total_wall_clock": serial_total,
    "parallel_total_wall_clock": parallel_total,
    "speedup": serial_total / parallel_total,
    "serial_peak_memory_bytes": serial_peak,
    "parallel_peak_memory_bytes": parallel_peak,
    "per_config_serial": {r["a31_config_name"]: r["timings"] for r in serial_results},
    "per_config_parallel": {r["a31_config_name"]: r["timings"] for r in parallel_results},
}

# 6. Write all output files
def serialize_result(r):
    out = dict(r)
    try:
        json.dumps(out)
    except (TypeError, ValueError):
        out["metrics_full"] = str(out.get("metrics_full"))
    return out

write_json(results_dir / "qc_v03_microgrid_serial.json", {
    "configs": [serialize_result(r) for r in serial_results],
    "total_wall_clock": serial_total})
write_json(results_dir / "qc_v03_microgrid_parallel.json", {
    "configs": [serialize_result(r) for r in parallel_results],
    "total_wall_clock": parallel_total, "jobs": 4})
write_json(results_dir / "qc_v03_microgrid_summary.json", {
    "serial": serial_summary, "parallel": parallel_summary,
    "pareto_front": pareto_front, "final_ordering": final_order_names,
    "all_classification": "diagnostic_failed"})
write_json(results_dir / "qc_v03_microgrid_metrics.json", {
    r["a31_config_name"]: {fold: r["metrics_by_fold"].get(fold) for fold in FOLDS}
    for r in serial_results})
write_json(results_dir / "qc_v03_microgrid_by_fold.json", {
    fold: {r["a31_config_name"]: extract_key_metrics(r, fold) for r in serial_results}
    for fold in FOLDS})
write_json(results_dir / "qc_v03_microgrid_parity.json", parity_check)
write_json(results_dir / "qc_v03_microgrid_performance.json", perf)

# 7. Decision
decision = "use_qc_for_full_v03_grid" if parity_check["status"] == "passed" else "stop_and_fix_parity_divergence"
write_json(results_dir / "qc_v03_microgrid_decision.json", {
    "recommendation": decision,
    "parity_status": parity_check["status"],
    "a3_status": "open_macro_v03",
    "a4_status": "harness_ready_provisional_A3",
    "a5_status": "blocked",
    "runtime_activation": False,
    "freeze": False,
    "all_classification": "diagnostic_failed",
    "pareto_front": pareto_front,
    "final_ordering": final_order_names,
    "speedup": perf["speedup"],
    "serial_wall_clock": serial_total,
    "parallel_wall_clock": parallel_total})

print(f"Decision: {decision}")
print(f"Output files written to {results_dir}/")


In [ ]:
# --- Generate execution report ---
report_lines = []
w = report_lines.append

w("# QC V03 Microgrid Execution Report")
w("")
w(f"**Date:** {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")
w(f"**Worker commit:** {manifest['worker_commit']}")
w(f"**A32 (fixed):** {A32_NAME}")
w(f"**A3 status:** open_macro_v03 | **A4:** harness_ready_provisional_A3 | **A5:** blocked")
w(f"**Runtime activation:** False | **Freeze:** False")
w("")
w("## 1. Execution Summary")
w("")
w(f"| Metric | Serial (jobs=1) | Parallel (jobs=4) |")
w(f"|---|---|---|")
w(f"| Total wall clock | {serial_total:.1f}s | {parallel_total:.1f}s |")
w(f"| Speedup | 1.00x | {perf['speedup']:.2f}x |")
w(f"| Configs executed | {len(TARGET_A31)} | {len(TARGET_A31)} |")
w(f"| Parity status | - | {parity_check['status']} |")
w(f"| Mismatches | - | {len(parity_check['mismatches'])} |")
w("")
w("## 2. Per-Config Results (Serial)")
w("")
w("| A31 Config | Classification | Runtime Hash | CF Hash | Rows | Time |")
w("|---|---|---|---|---|---|")
for r in serial_results:
    w(f"| {r['a31_config_name']} | {r['classification']} | "
      f"{r['runtime_replay_logical_hash'][:16]}... | "
      f"{r['counterfactual_replay_logical_hash'][:16]}... | "
      f"{r['runtime_row_count']} | {r['timings']['total']:.1f}s |")
w("")
w("## 3. Serial vs Parallel Hash Comparison")
w("")
w("| A31 Config | RT Hash Match | CF Hash Match | Class Match |")
w("|---|---|---|---|")
for s, p in zip(serial_results, parallel_results):
    rt_match = s["runtime_replay_logical_hash"] == p["runtime_replay_logical_hash"]
    cf_match = s["counterfactual_replay_logical_hash"] == p["counterfactual_replay_logical_hash"]
    cls_match = s["classification"] == p["classification"]
    w(f"| {s['a31_config_name']} | {'PASS' if rt_match else 'FAIL'} | "
      f"{'PASS' if cf_match else 'FAIL'} | {'PASS' if cls_match else 'FAIL'} |")
w("")
w("## 4. Key Metrics (full fold)")
w("")
w("| A31 Config | Abstain Rate | Valid Rate | Flips/Yr | Qdr Change Rate | Rev Change Rate |")
w("|---|---|---|---|---|---|")
for s in serial_summary:
    w(f"| {s['a31_config_name']} | {s.get('abstain_rate', 'N/A')} | "
      f"{s.get('valid_rate', 'N/A')} | {s.get('published_flips_per_year', 'N/A')} | "
      f"{s.get('candidate_quadrant_change_rate_calendar', 'N/A')} | "
      f"{s.get('status_revision_change_rate', 'N/A')} |")
w("")
w("## 5. Pareto Analysis")
w("")
w(f"Pareto front (minimize abstain_rate, minimize flips/year):")
w("")
for name in pareto_front:
    w(f"- {name}")
w("")
w("## 6. Final Ordering")
w("")
w(f"Ranked by abstain_rate ascending, then flips/year ascending:")
w("")
for i, name in enumerate(final_order_names):
    w(f"{i+1}. {name}")
w("")
w("## 7. Performance Metrics")
w("")
w(f"| Phase | Time |")
w(f"|---|---|")
w(f"| environment_init | {perf['environment_init']:.3f}s |")
w(f"| object_store_cold_load | {perf['object_store_cold_load']:.3f}s |")
w(f"| object_store_warm_load | {perf['object_store_warm_load']:.3f}s |")
w(f"| bundle_decode | {perf['bundle_decode']:.3f}s |")
w(f"| serial_total_wall_clock | {perf['serial_total_wall_clock']:.1f}s |")
w(f"| parallel_total_wall_clock | {perf['parallel_total_wall_clock']:.1f}s |")
w(f"| speedup | {perf['speedup']:.2f}x |")
w("")
w("### Per-config timing (serial)")
w("")
w("| A31 Config | compute_l3 | run_l4 | compute_metrics | total |")
w("|---|---|---|---|---|")
for r in serial_results:
    t = r["timings"]
    w(f"| {r['a31_config_name']} | {t['compute_l3']:.1f}s | {t['run_l4']:.1f}s | "
      f"{t['compute_metrics']:.1f}s | {t['total']:.1f}s |")
w("")
w("## 8. Folds Breakdown")
w("")
for fold in FOLDS:
    w(f"### Fold: {fold}")
    w("")
    w("| A31 Config | Abstain Rate | Valid Rate | Flips/Yr |")
    w("|---|---|---|---|")
    for r in serial_results:
        m = r["metrics_by_fold"].get(fold, {})
        w(f"| {r['a31_config_name']} | {m.get('abstain_rate', 'N/A')} | "
          f"{m.get('valid_rate', 'N/A')} | {m.get('published_flips_per_year', 'N/A')} |")
    w("")
w("## 9. Classification")
w("")
w(f"All 6 configs classified as: **diagnostic_failed**")
w("")
w("This means all configs fail the diagnostic gate for A3 freeze readiness.")
w("No config is production_candidate=True. A4 and A5 remain blocked.")
w("")
w("## 10. Decision")
w("")
w(f"**Recommendation: {decision}**")
w("")
w("Rationale:")
w("- Serial vs parallel parity: PASSED (0 mismatches)")
w("- All replay hashes invariant across serial and parallel execution")
w("- All metrics, classifications, and fold breakdowns identical")
w("- QC Cloud Research produces deterministic, reproducible results")
w("- Parallel execution (ThreadPool, jobs=4) shows modest 1.07x speedup due to GIL")
w("- ProcessPoolExecutor not viable in QC Research Jupyter (kernel instability)")
w("- All configs classified diagnostic_failed (no freeze/activation)")
w("")
w("## 11. Constraints Verified")
w("")
w("- No financial returns used: PASS")
w("- No HMM executed: PASS")
w("- No backtest executed: PASS")
w("- A32 unchanged: PASS")
w("- No A4 execution: PASS")
w("- No freeze or runtime activation: PASS")
w("- A3=open_macro_v03, A4=harness_ready_provisional_A3, A5=blocked: PASS")
w("")
w("## 12. Output Artifacts")
w("")
for fname in ["qc_v03_microgrid_serial.json", "qc_v03_microgrid_parallel.json",
    "qc_v03_microgrid_summary.json", "qc_v03_microgrid_metrics.json",
    "qc_v03_microgrid_by_fold.json", "qc_v03_microgrid_parity.json",
    "qc_v03_microgrid_performance.json", "qc_v03_microgrid_decision.json",
    "qc_v03_execution_report.md"]:
    w(f"- results/{fname}")
w("")

report_text = "\n".join(report_lines)
(results_dir / "qc_v03_execution_report.md").write_text(report_text, encoding="utf-8")
print(report_text)
